# Chapter 72: Security Log Analysis & SIEM Integration

**Volume 4 — Security Operations**

**Your network generates 50,000 security events every day.**
Your team can investigate 100. That is 0.2% coverage.
The attacker operated in the other 99.8% — and you never knew.

The answer is not more analysts. It is a smarter pipeline.

### What You Will Build — 5 SIEM Layers

| Demo | Layer | What It Does |
|------|-------|-------------|
| **1. Log Normalizer** | Collection + Normalization | Parse Cisco ASA / Juniper / Palo Alto / Windows into one schema |
| **2. CEP Correlation Engine** | Correlation | Stateful pattern matching: brute force, port scan, impossible travel |
| **3. Attack Chain Reconstructor** | Temporal Correlation | Expand around an anchor event to rebuild the full attack timeline |
| **4. Threat Intel Enrichment** | Enrichment | STIX-like IoC matching + RAG threat context per alert |
| **5. Full SIEM Pipeline + LLM Brief** | AI Triage | All layers wired — 50,000 events in, 50 prioritized incidents out |

### The Architecture Insight
> **No single log source is sufficient. A firewall sees network patterns.
> Auth logs see identity. Endpoint logs see process behavior.
> SIEM correlates all four — and AI narrates what the correlation found,
> so an analyst reads a 2-minute incident brief instead of 90 minutes of raw logs.**

In [ ]:
# pip install anthropic
# SIEM Integration — pure Python, no external SIEM required!

import os, re, json, time, math, hashlib
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

# ── Anthropic client ───────────────────────────────────────────────────────────
api_key = os.environ.get("ANTHROPIC_API_KEY", "")
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    '''Call Anthropic API or return a simulation response.'''
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251011",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    # Simulation responses based on keywords
    if "brute force" in prompt.lower() or "failed login" in prompt.lower():
        return ("MITRE T1110 (Brute Force). Pattern: 5 failed SSH attempts then success "
                "from same IP. Recommend: block source IP, audit accessed account, "
                "check for lateral movement within 30 min.")
    if "port scan" in prompt.lower() or "reconnaissance" in prompt.lower():
        return ("MITRE T1046 (Network Service Discovery). Sequential port probing across "
                "22/80/443/445/3389. Recommend: block scanning IP, check if any port "
                "accepted a connection, review firewall rules for those ports.")
    if "exfiltration" in prompt.lower() or "4.7gb" in prompt.lower():
        return ("MITRE T1048 (Exfiltration Over Alternative Protocol). Large SCP transfer "
                "from internal DB server after lateral movement. CRITICAL. Recommend: "
                "isolate source host, preserve memory image, notify IR team immediately.")
    if "attack chain" in prompt.lower() or "timeline" in prompt.lower():
        return ("Attack chain reconstructed: Initial recon (T1046) -> Brute force SSH "
                "(T1110) -> Successful auth -> Lateral movement to DB (T1021) -> "
                "Data staging -> Exfiltration (T1048). Total dwell time: 44 minutes. "
                "Scope: 2 hosts compromised. Severity: CRITICAL.")
    return ("Anomalous activity detected. Multi-stage attack pattern consistent with "
            "targeted intrusion. Recommend analyst review and CISO notification.")

print("Setup complete. All SIEM layers ready.")
print(f"Mode: {'API' if USE_API else 'Simulation'}")


## Demo 1: Multi-Vendor Log Normalizer (CIM Schema)

The first problem every SIEM solves: **Cisco ASA logs look nothing like Juniper SRX logs**,
which look nothing like Palo Alto logs — even though all three record the same event
(a firewall deny).

Without normalization, you need separate correlation rules for each vendor.
With normalization, one rule covers all vendors.

**Common Information Model (CIM)** — vendor-neutral fields:

| CIM Field | Cisco ASA Example | Juniper SRX Example | Windows Event |
|-----------|------------------|--------------------|----|
| `src_ip` | `src outside:1.2.3.4/52000` | `RT_FLOW_DENY src=1.2.3.4` | `IpAddress: 1.2.3.4` |
| `dst_port` | `dst inside:10.0.0.1/443` | `dst-port=443` | `DestPort: 443` |
| `action` | `Deny` / `Permit` | `DENY` / `ALLOW` | `Success` / `Failure` |

*Network analogy: log normalization is like standard interface naming conventions.
When you have Cisco, Juniper, and Arista in the same fabric, you establish naming
standards so any engineer reads any device instantly. CIM does the same for log fields.*

In [ ]:
# ── Demo 1: Multi-Vendor Log Normalizer ───────────────────────────────────────

@dataclass
class NormalizedEvent:
    '''Common Information Model — vendor-neutral security event schema.'''
    timestamp: str
    src_ip: str
    dst_ip: str
    dst_port: int
    action: str       # "allow" or "deny"
    protocol: str     # "tcp", "udp", "icmp"
    event_type: str   # "firewall", "auth", "dns", "endpoint"
    source: str       # which device/vendor produced this log
    raw: str = ""     # original log line for audit trail
    username: str = ""
    outcome: str = "" # for auth: "success" or "failure"

# ── Parser 1: Cisco ASA ───────────────────────────────────────────────────────
# Example: %ASA-4-106023: Deny tcp src outside:185.220.1.100/43221 dst inside:10.0.0.5/22
CISCO_ASA_RE = re.compile(
    r'%ASA-\d-\d+: (Deny|Permit) (\w+) src \S+:([\d.]+)/\d+ dst \S+:([\d.]+)/(\d+)'
)

def parse_cisco_asa(line: str, ts: str = "T00:00:00") -> Optional[NormalizedEvent]:
    '''Parse Cisco ASA firewall log line into CIM schema.'''
    m = CISCO_ASA_RE.search(line)
    if not m:
        return None
    action_raw, proto, src, dst, port = m.groups()
    return NormalizedEvent(
        timestamp=ts, src_ip=src, dst_ip=dst, dst_port=int(port),
        action="deny" if action_raw == "Deny" else "allow",
        protocol=proto.lower(), event_type="firewall", source="cisco-asa", raw=line
    )

# ── Parser 2: Juniper SRX ─────────────────────────────────────────────────────
# Example: RT_FLOW_DENY: session denied 1.2.3.4/43221->10.0.0.5/22 proto TCP
JUNIPER_SRX_RE = re.compile(
    r'RT_FLOW_(DENY|ALLOW): session \S+ ([\d.]+)/\d+->([\d.]+)/(\d+) proto (\w+)'
)

def parse_juniper_srx(line: str, ts: str = "T00:00:00") -> Optional[NormalizedEvent]:
    '''Parse Juniper SRX flow log line into CIM schema.'''
    m = JUNIPER_SRX_RE.search(line)
    if not m:
        return None
    action_raw, src, dst, port, proto = m.groups()
    return NormalizedEvent(
        timestamp=ts, src_ip=src, dst_ip=dst, dst_port=int(port),
        action="deny" if action_raw == "DENY" else "allow",
        protocol=proto.lower(), event_type="firewall", source="juniper-srx", raw=line
    )

# ── Parser 3: Windows Event Log 4624/4625 (auth) ─────────────────────────────
# Example: EventID=4625 Account=jsmith IpAddress=185.220.1.100 Status=0xC000006A
WINDOWS_AUTH_RE = re.compile(
    r'EventID=(\d+) Account=(\S+) IpAddress=([\d.]+)(?:.* Status=(\S+))?'
)

def parse_windows_auth(line: str, ts: str = "T00:00:00") -> Optional[NormalizedEvent]:
    '''Parse Windows Security Event Log auth record into CIM schema.'''
    m = WINDOWS_AUTH_RE.search(line)
    if not m:
        return None
    evid, user, src_ip, status = m.groups()
    # 4624 = success, 4625 = failure
    success = (evid == "4624")
    return NormalizedEvent(
        timestamp=ts, src_ip=src_ip, dst_ip="10.0.0.5", dst_port=0,
        action="allow" if success else "deny",
        protocol="kerberos", event_type="auth", source="windows-ad",
        username=user, outcome="success" if success else "failure",
        raw=line
    )

# ── Parser 4: Palo Alto CSV ───────────────────────────────────────────────────
# Example: 2026-02-26T03:14,185.220.1.100,10.0.0.5,8080,deny,tcp
def parse_paloalto_csv(line: str) -> Optional[NormalizedEvent]:
    '''Parse Palo Alto CSV log format into CIM schema.'''
    parts = line.split(",")
    if len(parts) < 6:
        return None
    try:
        return NormalizedEvent(
            timestamp=parts[0], src_ip=parts[1].strip(), dst_ip=parts[2].strip(),
            dst_port=int(parts[3].strip()),
            action=parts[4].strip().lower(), protocol=parts[5].strip().lower(),
            event_type="firewall", source="palo-alto", raw=line
        )
    except (ValueError, IndexError):
        return None

# ── Master normalizer ─────────────────────────────────────────────────────────
def normalize(line: str, ts: str = "T00:00:00") -> Optional[NormalizedEvent]:
    '''Try all parsers in order — return first match.'''
    return (parse_cisco_asa(line, ts) or
            parse_juniper_srx(line, ts) or
            parse_windows_auth(line, ts) or
            parse_paloalto_csv(line))

# ── Raw logs from 4 different vendors ─────────────────────────────────────────
raw_logs = [
    ("T03:10:00", "%ASA-4-106023: Deny tcp src outside:185.220.1.100/43221 dst inside:10.0.0.5/22"),
    ("T03:10:05", "%ASA-4-106023: Deny tcp src outside:185.220.1.100/43230 dst inside:10.0.0.5/22"),
    ("T03:10:10", "%ASA-4-106023: Deny tcp src outside:185.220.1.100/43238 dst inside:10.0.0.5/3389"),
    ("T03:10:15", "RT_FLOW_DENY: session denied 185.220.1.100/51000->10.0.0.5/445 proto TCP"),
    ("T03:10:20", "RT_FLOW_ALLOW: session allowed 185.220.1.100/51001->10.0.0.5/443 proto TCP"),
    ("T03:11:00", "EventID=4625 Account=admin IpAddress=185.220.1.100 Status=0xC000006A"),
    ("T03:11:05", "EventID=4625 Account=admin IpAddress=185.220.1.100 Status=0xC000006A"),
    ("T03:11:10", "EventID=4624 Account=admin IpAddress=185.220.1.100 Status=0x0"),
    ("T03:12:00", "2026-02-26T03:12,185.220.1.100,10.0.0.20,445,deny,tcp"),
]

print("=" * 65)
print("LOG NORMALIZER — MULTI-VENDOR TO CIM SCHEMA")
print("=" * 65)
print(f"{'Time':<10} {'Source':<14} {'Src IP':<18} {'Dst':>18} {'Action':<8} {'Type'}")
print("-" * 65)

normalized_events = []
for ts, line in raw_logs:
    ev = normalize(line, ts)
    if ev:
        normalized_events.append(ev)
        dst_str = f"{ev.dst_ip}:{ev.dst_port}" if ev.dst_port else ev.dst_ip
        user_str = f" [{ev.username}]" if ev.username else ""
        print(f"{ev.timestamp:<10} {ev.source:<14} {ev.src_ip:<18} {dst_str:>18} "
              f"{ev.action:<8} {ev.event_type}{user_str}")
    else:
        print(f"  [WARN] Could not parse: {line[:60]}")

vendors = len(set(e.source for e in normalized_events))
print(f"\n[Normalizer] {len(normalized_events)} events normalized from {vendors} vendors.")
print("All events now share identical field names — one correlation rule covers all vendors.")


## Demo 2: CEP Correlation Engine (Stateful Pattern Matching)

**CEP (Complex Event Processing)** is the secret behind SIEM correlation rules.

The key inversion from databases: in a database, data is stored and queries are transient.
In a CEP engine, **queries (rules) are persistent — data flows through them**.

Think of it like a BGP route filter: the route map sits there waiting.
When a BGP UPDATE arrives, the route map checks conditions and acts.
CEP does the same for security events — rules wait, events flow.

**Three rules we implement:**

| Rule | Pattern | What It Catches |
|------|---------|----------------|
| **Brute Force** | 4+ failed auths from same IP in 5 min | SSH/RDP password guessing |
| **Port Scan** | 5+ deny events from same IP in 2 min | Reconnaissance / discovery |
| **Impossible Travel** | Auth success from 2 IPs >1000km apart in <1hr | Credential theft in use |

Each rule maintains **state** — it remembers partial matches and waits for completion.

In [ ]:
# ── Demo 2: CEP Correlation Engine ────────────────────────────────────────────

@dataclass
class CorrelationAlert:
    '''Fired when a CEP rule pattern is completed.'''
    rule_name: str
    severity: str
    entity: str          # the IP or user at the center of the pattern
    events: list         # the events that triggered this alert
    description: str
    score: float

class CEPEngine:
    '''
    Simple stateful CEP (Complex Event Processing) engine.
    Rules are registered as standing queries.
    Events flow through and complete partial matches.
    '''

    def __init__(self, window_seconds: int = 300):
        self.window_sec = window_seconds
        # State stores: keyed by entity (IP or user)
        # Each stores deque of (timestamp_float, event) tuples
        self.state: Dict[str, deque] = defaultdict(lambda: deque(maxlen=100))
        self.alerts: List[CorrelationAlert] = []
        self._base_ts = time.time() - 3600   # simulate events 1 hour ago

    def _ts_to_float(self, ts_str: str) -> float:
        '''Convert T03:10:00 style timestamp to float offset from base.'''
        parts = ts_str.lstrip("T").split(":")
        h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
        return self._base_ts + h * 3600 + m * 60 + s

    def _prune_window(self, entity: str, now: float):
        '''Remove events older than the sliding window.'''
        q = self.state[entity]
        while q and (now - q[0][0]) > self.window_sec:
            q.popleft()

    def ingest(self, event: NormalizedEvent) -> Optional[CorrelationAlert]:
        '''Process one event through all CEP rules. Returns alert if pattern fires.'''
        now = self._ts_to_float(event.timestamp)
        entity = event.src_ip   # correlate by source IP

        self._prune_window(entity, now)
        self.state[entity].append((now, event))

        # Rule 1: Brute Force — 4+ failed auth from same IP in window
        alert = self._rule_brute_force(entity, now)
        if alert:
            self.alerts.append(alert)
            return alert

        # Rule 2: Port Scan — 5+ deny events from same IP in window
        alert = self._rule_port_scan(entity, now)
        if alert:
            self.alerts.append(alert)
            return alert

        return None

    def _rule_brute_force(self, entity: str, now: float) -> Optional[CorrelationAlert]:
        '''Fires when 4+ auth failures from same IP within the window.'''
        failed_auths = [
            ev for ts, ev in self.state[entity]
            if ev.event_type == "auth" and ev.outcome == "failure"
        ]
        if len(failed_auths) >= 4:
            # Only fire once — check if we already alerted this entity recently
            if any(a.entity == entity and a.rule_name == "BRUTE_FORCE"
                   for a in self.alerts[-5:]):
                return None
            return CorrelationAlert(
                rule_name="BRUTE_FORCE",
                severity="HIGH",
                entity=entity,
                events=failed_auths,
                description=(f"Brute force: {len(failed_auths)} failed auth attempts "
                             f"from {entity} in {self.window_sec//60}min window"),
                score=0.82,
            )
        return None

    def _rule_port_scan(self, entity: str, now: float) -> Optional[CorrelationAlert]:
        '''Fires when 5+ firewall denies from same IP to different ports within window.'''
        denied = [
            ev for ts, ev in self.state[entity]
            if ev.event_type == "firewall" and ev.action == "deny"
        ]
        unique_ports = len(set(ev.dst_port for ev in denied))
        if unique_ports >= 4 and len(denied) >= 5:
            if any(a.entity == entity and a.rule_name == "PORT_SCAN"
                   for a in self.alerts[-5:]):
                return None
            return CorrelationAlert(
                rule_name="PORT_SCAN",
                severity="MEDIUM",
                entity=entity,
                events=denied,
                description=(f"Port scan: {len(denied)} firewall denies from {entity} "
                             f"across {unique_ports} unique ports"),
                score=0.68,
            )
        return None

# ── Run the CEP engine on our normalized events ────────────────────────────────
print("=" * 60)
print("CEP CORRELATION ENGINE — STATEFUL PATTERN MATCHING")
print("=" * 60)
print(f"Window size: 5 minutes | Processing {len(normalized_events)} events\n")

engine = CEPEngine(window_seconds=300)
fired_alerts = []

for ev in normalized_events:
    alert = engine.ingest(ev)
    if alert:
        print(f"[RULE FIRED] {alert.rule_name} | {alert.severity} | entity={alert.entity}")
        print(f"  {alert.description}")
        print(f"  Score: {alert.score:.2f}")
        # LLM enrichment
        analysis = llm_analyze(
            f"CEP rule fired: {alert.rule_name} severity={alert.severity}. "
            f"{alert.description}. MITRE technique? Recommended analyst action? "
            f"Under 70 words.",
            max_tokens=100
        )
        print(f"  LLM: {analysis}\n")
        fired_alerts.append(alert)

if not fired_alerts:
    print("  No patterns completed in this event window.")

print(f"[CEP] {len(normalized_events)} events processed -> {len(fired_alerts)} alerts fired.")
print("Rules persisted, ready for next event batch.")


## Demo 3: Multi-Stage Attack Chain Reconstructor

Individual events are rarely alarming alone.
A failed login at 3am is noise.
That same failed login + 4 more failures + 1 success + lateral movement to DB + 4.7GB SCP transfer
is a **critical incident** — but you only see it by connecting events across time.

**Temporal Correlation — 5 steps:**

```
Step 1: Anchor event fires   (CEP rule or anomaly alert)
Step 2: Expand BACKWARD      (what happened from same source before the anchor?)
Step 3: Expand FORWARD       (what happened after the anchor?)
Step 4: Lateral pivot        (if source accessed new hosts, follow those hosts too)
Step 5: LLM synthesizes      (narrative: timeline, MITRE chain, scope, severity)
```

The LLM's role is NOT detection (that was CEP). The LLM reads the correlated
timeline and writes an analyst-readable incident brief in 2 minutes,
replacing what used to take 90 minutes of manual log correlation.

In [ ]:
# ── Demo 3: Attack Chain Reconstructor ────────────────────────────────────────

@dataclass
class SecurityEvent:
    '''Extended event type for attack chain reconstruction.'''
    timestamp: str
    ts_offset: float      # seconds from epoch for sorting
    src_ip: str
    dst_ip: str
    dst_port: int
    action: str
    event_type: str
    source: str
    description: str
    username: str = ""

class AttackChainReconstructor:
    '''
    Given an anchor alert, expands backward and forward in time
    to reconstruct the full attack chain across all log sources.
    '''

    def __init__(self, all_events: List[SecurityEvent], window_seconds: int = 3600):
        self.events = all_events
        self.window = window_seconds

    def reconstruct(self, anchor: SecurityEvent) -> dict:
        '''
        Reconstruct attack chain around an anchor event.
        Returns: {"timeline": [...], "hosts_involved": [...], "ips_involved": [...]}
        '''
        anchor_ts = anchor.ts_offset

        # Step 1: Gather events from same source IP in +/- window
        related = [
            ev for ev in self.events
            if ev.src_ip == anchor.src_ip
            and abs(ev.ts_offset - anchor_ts) <= self.window
        ]

        # Step 2: Lateral pivot — if the attacker reached a new host,
        # gather events where that host is the SOURCE (they used it to move)
        pivot_ips = set(ev.dst_ip for ev in related if ev.action == "allow")
        pivot_ips.discard(anchor.src_ip)

        for pivot_ip in pivot_ips:
            pivot_events = [
                ev for ev in self.events
                if (ev.src_ip == pivot_ip or ev.dst_ip == pivot_ip)
                and ev.ts_offset >= anchor_ts
                and ev.ts_offset <= anchor_ts + self.window
            ]
            related.extend(pivot_events)

        # Deduplicate and sort by time
        seen = set()
        unique_related = []
        for ev in related:
            key = (ev.timestamp, ev.src_ip, ev.dst_ip, ev.dst_port)
            if key not in seen:
                seen.add(key)
                unique_related.append(ev)
        unique_related.sort(key=lambda e: e.ts_offset)

        hosts_involved = list(set(
            [ev.src_ip for ev in unique_related] +
            [ev.dst_ip for ev in unique_related if ev.dst_ip != "unknown"]
        ))
        return {
            "anchor": anchor,
            "timeline": unique_related,
            "hosts_involved": hosts_involved,
            "pivot_ips": list(pivot_ips),
        }

    def llm_incident_brief(self, chain: dict) -> str:
        '''Submit the reconstructed chain to the LLM for incident narrative.'''
        timeline_text = "\n".join(
            f"  {ev.timestamp} | {ev.src_ip}->{ev.dst_ip}:{ev.dst_port} "
            f"[{ev.action.upper()}] {ev.description}"
            for ev in chain["timeline"]
        )
        prompt = (
            f"Security incident attack chain reconstruction:\n"
            f"Anchor: {chain['anchor'].description}\n"
            f"Hosts involved: {chain['hosts_involved']}\n\n"
            f"Full timeline:\n{timeline_text}\n\n"
            f"Write an incident brief: 1) What happened (narrative), "
            f"2) MITRE ATT&CK kill chain, 3) Scope (which hosts/accounts), "
            f"4) Severity, 5) Immediate actions. Under 120 words."
        )
        return llm_analyze(prompt, max_tokens=160)

# ── Simulate a full attack event log ──────────────────────────────────────────
base = time.time() - 7200   # 2 hours ago
all_security_events = [
    SecurityEvent("T03:10:00", base+0,    "185.220.1.100","10.0.0.5",  22, "deny",  "firewall","cisco-asa",  "FW DENY ssh to bastion"),
    SecurityEvent("T03:10:05", base+5,    "185.220.1.100","10.0.0.5",  22, "deny",  "firewall","cisco-asa",  "FW DENY ssh to bastion"),
    SecurityEvent("T03:10:10", base+10,   "185.220.1.100","10.0.0.5",  3389,"deny", "firewall","cisco-asa",  "FW DENY rdp to bastion"),
    SecurityEvent("T03:11:00", base+60,   "185.220.1.100","10.0.0.5",  0,  "deny",  "auth",    "windows-ad", "Auth FAIL admin from external", username="admin"),
    SecurityEvent("T03:11:05", base+65,   "185.220.1.100","10.0.0.5",  0,  "deny",  "auth",    "windows-ad", "Auth FAIL admin from external", username="admin"),
    SecurityEvent("T03:11:10", base+70,   "185.220.1.100","10.0.0.5",  0,  "deny",  "auth",    "windows-ad", "Auth FAIL admin from external", username="admin"),
    SecurityEvent("T03:11:15", base+75,   "185.220.1.100","10.0.0.5",  0,  "deny",  "auth",    "windows-ad", "Auth FAIL admin from external", username="admin"),
    SecurityEvent("T03:11:22", base+82,   "185.220.1.100","10.0.0.5",  22, "allow", "auth",    "linux-ssh",  "SSH SUCCESS admin -> bastion-01", username="admin"),
    SecurityEvent("T03:18:00", base+480,  "10.0.0.5",    "10.0.1.20", 22, "allow", "endpoint","sysmon",     "Lateral: bastion-01 -> db-server-02 SSH"),
    SecurityEvent("T03:31:00", base+1260, "10.0.0.5",    "10.0.1.20", 0,  "allow", "endpoint","sysmon",     "SCP 4.7GB from db-server-02 to bastion-01"),
    SecurityEvent("T03:45:00", base+2100, "185.220.1.100","8.8.8.8",  443,"allow", "firewall","palo-alto",  "Outbound HTTPS from external IP"),
    # Normal background noise (different IP)
    SecurityEvent("T03:10:30", base+30,   "192.168.1.50","10.0.0.1",  80, "allow", "firewall","cisco-asa",  "Internal web browsing - normal"),
]

print("=" * 65)
print("ATTACK CHAIN RECONSTRUCTOR — TEMPORAL CORRELATION")
print("=" * 65)

# Anchor = the brute force success event
anchor_event = all_security_events[7]   # SSH SUCCESS at T03:11:22
print(f"Anchor event: {anchor_event.description} at {anchor_event.timestamp}\n")

reconstructor = AttackChainReconstructor(all_security_events, window_seconds=3000)
chain = reconstructor.reconstruct(anchor_event)

print(f"Events in chain: {len(chain['timeline'])}")
print(f"Hosts involved: {chain['hosts_involved']}")
print(f"Lateral pivots: {chain['pivot_ips']}")
print("\n[RECONSTRUCTED TIMELINE]")
for ev in chain["timeline"]:
    marker = " <-- ANCHOR" if ev == anchor_event else ""
    print(f"  {ev.timestamp} | {ev.src_ip}->{ev.dst_ip} | {ev.action.upper()} | {ev.description}{marker}")

print("\n[LLM INCIDENT BRIEF]")
brief = reconstructor.llm_incident_brief(chain)
print(brief)


## Demo 4: Threat Intelligence Enrichment (STIX-like IoC Matching)

A security event processed in isolation has limited context.
The same event **enriched with threat intelligence** can be immediately actionable.

**STIX (Structured Threat Information eXpression)** = standardized format for IoCs:
- Malicious IP addresses, domains, file hashes
- Threat actor profiles, malware families, attack patterns
- Relationships: `threat-actor USES malware`, `malware INDICATES attack-pattern`

**TAXII** = the transport protocol (like BGP for threat intelligence).
Your SIEM subscribes to a TAXII feed and receives STIX updates automatically.

When an alert fires, the enrichment pipeline:
1. Looks up source IP / domain / hash against the IoC index
2. Returns threat actor attribution, campaign, severity
3. RAG-style lookup: queries threat report text for additional context

```
Alert: src_ip=185.220.1.100
IoC lookup: MATCH -> Lazarus Group, APT38, targeting financial sector
RAG context: "This IP was observed in a campaign targeting SWIFT systems in Q4 2025"
Combined: analyst sees everything, no manual VirusTotal search needed
```

In [ ]:
# ── Demo 4: Threat Intelligence Enrichment ────────────────────────────────────

# Simulated STIX indicator objects (in production: subscribe to TAXII feed)
STIX_INDICATORS = {
    # IP indicators
    "185.220.1.100": {
        "type": "ipv4-addr",
        "threat_actor": "Lazarus Group",
        "campaign": "Operation AppleJeus 2025",
        "malware_family": "BLINDINGCAN",
        "confidence": 87,
        "last_seen": "2026-01-15",
        "tags": ["apt", "financial-sector", "north-korea"],
    },
    "91.108.4.12": {
        "type": "ipv4-addr",
        "threat_actor": "APT29 (Cozy Bear)",
        "campaign": "SolarStorm Successor",
        "malware_family": "SUNBURST-variant",
        "confidence": 72,
        "last_seen": "2026-02-01",
        "tags": ["apt", "government", "russia"],
    },
    "203.0.113.99": {
        "type": "ipv4-addr",
        "threat_actor": "Generic cybercrime",
        "campaign": "Ransomware-as-a-Service",
        "malware_family": "LockBit 3.0",
        "confidence": 91,
        "last_seen": "2026-02-20",
        "tags": ["ransomware", "crimeware"],
    },
    # Domain indicators
    "xkqpzbmvowf.ru": {
        "type": "domain-name",
        "threat_actor": "Lazarus Group",
        "campaign": "Operation AppleJeus 2025",
        "malware_family": "BLINDINGCAN C2",
        "confidence": 94,
        "last_seen": "2026-02-24",
        "tags": ["c2", "dga", "apt"],
    },
}

# Simulated threat report corpus for RAG-style enrichment
THREAT_REPORTS = [
    {
        "id": "RPT-001",
        "title": "Lazarus Group Financial Sector Campaign Q1 2026",
        "content": ("Lazarus Group (DPRK-nexus APT) has been conducting targeted "
                    "intrusions against financial institutions using spearphishing and "
                    "credential stuffing. Infrastructure includes Tor exit nodes in the "
                    "185.220.0.0/16 range. Post-compromise behavior: SSH brute force "
                    "followed by lateral movement to database servers and SWIFT terminal access."),
        "iocs": ["185.220.1.100", "185.220.1.101"],
    },
    {
        "id": "RPT-002",
        "title": "LockBit 3.0 Affiliate TTPs",
        "content": ("LockBit 3.0 affiliates use RDP brute force as primary initial access. "
                    "Dwell time averages 5 days before encryption. Pre-encryption: "
                    "exfiltration via rclone to cloud storage, BloodHound AD enumeration, "
                    "Cobalt Strike beacon over HTTPS."),
        "iocs": ["203.0.113.99"],
    },
]

class ThreatIntelEnricher:
    '''
    Enriches security alerts with STIX-like IoC data and RAG threat report context.
    '''

    def __init__(self, ioc_db: dict, reports: list):
        self.ioc_db = ioc_db
        self.reports = reports

    def lookup_ioc(self, indicator: str) -> Optional[dict]:
        '''Exact lookup of IP, domain, or hash in IoC database.'''
        return self.ioc_db.get(indicator)

    def rag_lookup(self, indicator: str) -> Optional[str]:
        '''Simple keyword-based RAG: find threat reports mentioning this indicator.'''
        for report in self.reports:
            if indicator in report.get("iocs", []) or indicator in report["content"]:
                return f"[{report['id']}] {report['title']}: {report['content'][:200]}..."
        return None

    def enrich_alert(self, alert_entity: str, alert_description: str) -> dict:
        '''
        Full enrichment pipeline:
        1. IoC exact match
        2. RAG threat report context
        3. LLM synthesis
        Returns enriched alert dict.
        '''
        ioc_match = self.lookup_ioc(alert_entity)
        rag_context = self.rag_lookup(alert_entity)

        enrichment = {
            "entity": alert_entity,
            "ioc_match": ioc_match,
            "rag_context": rag_context,
            "llm_summary": None,
        }

        if ioc_match:
            prompt = (
                f"Alert entity: {alert_entity}\n"
                f"Alert: {alert_description}\n"
                f"Threat intel match: {json.dumps(ioc_match)}\n"
                f"Threat report context: {rag_context or 'none'}\n\n"
                f"Write 3-sentence analyst briefing: who is this actor, "
                f"what are their typical next steps, what should the analyst do now?"
            )
            enrichment["llm_summary"] = llm_analyze(prompt, max_tokens=120)

        return enrichment

# ── Run enrichment on our attack events ───────────────────────────────────────
print("=" * 65)
print("THREAT INTEL ENRICHMENT — STIX IoC MATCHING + RAG CONTEXT")
print("=" * 65)

enricher = ThreatIntelEnricher(STIX_INDICATORS, THREAT_REPORTS)

entities_to_enrich = [
    ("185.220.1.100", "Brute force SSH then lateral movement detected"),
    ("10.0.0.5",      "Internal host accessed by attacker (bastion)"),    # no match expected
    ("203.0.113.99",  "Outbound C2 beacon detected via beacon detector"),
    ("xkqpzbmvowf.ru","DGA domain queried 847 times by compromised host"),
]

for entity, description in entities_to_enrich:
    result = enricher.enrich_alert(entity, description)
    print(f"\nEntity: {entity}")
    if result["ioc_match"]:
        ioc = result["ioc_match"]
        print(f"  [IOC MATCH] Actor={ioc['threat_actor']}  "
              f"Campaign={ioc['campaign']}  "
              f"Confidence={ioc['confidence']}%")
        print(f"  Tags: {ioc['tags']}")
        if result["rag_context"]:
            print(f"  RAG: {result['rag_context'][:120]}...")
        if result["llm_summary"]:
            print(f"  Brief: {result['llm_summary']}")
    else:
        print(f"  [NO MATCH] Entity not in threat intelligence database.")

print("\n[TI Enricher] Enrichment complete. All matched alerts elevated in priority queue.")


## Demo 5: Full SIEM Pipeline — 50,000 Events In, 50 Incidents Out

**All four layers wired together** — the complete production SIEM pipeline:

```
Raw logs (Cisco/Juniper/PaloAlto/Windows)
        |
        v
[Layer 1] Log Normalizer   -> CIM events
        |
        v
[Layer 2] CEP Engine       -> Correlation alerts
        |
        v
[Layer 3] Attack Chain     -> Temporal event expansion
        |
        v
[Layer 4] TI Enrichment    -> IoC match + RAG context
        |
        v
[Layer 5] LLM Triage       -> Prioritized incident brief
        |
        v
Analyst Queue (50-100 incidents/day, not 50,000 raw events)
```

**The before/after:**

| Metric | Before AI | After AI |
|--------|-----------|----------|
| Events per day | 50,000 | 50,000 |
| Analyst review rate | 0.2% | 100% (auto-triaged) |
| Context gathering time | 90 min/incident | 2 min (pre-populated) |
| Attack chain visible | Only if analyst correlates manually | Automatic |
| Threat actor attribution | Manual VirusTotal search | Automatic IoC match |

In [ ]:
# ── Demo 5: Full SIEM Pipeline ────────────────────────────────────────────────

@dataclass
class SIEMIncident:
    '''A fully enriched, prioritized incident ready for analyst review.'''
    incident_id: str
    priority: int           # 1=CRITICAL, 2=HIGH, 3=MEDIUM, 4=LOW
    severity: str
    entity: str
    rule_fired: str
    event_count: int
    threat_actor: Optional[str]
    ioc_confidence: int     # 0-100, 0 if no IoC match
    attack_chain_length: int
    llm_brief: str
    mitre_tags: List[str]

class SIEMPipeline:
    '''
    Complete SIEM pipeline:
    Normalization -> CEP -> Attack Chain -> TI Enrichment -> LLM Triage -> Priority Queue
    '''

    def __init__(self):
        self.normalizer_stats = {"total": 0, "parsed": 0, "failed": 0}
        self.cep_engine = CEPEngine(window_seconds=300)
        self.reconstructor = None   # initialized after collecting all events
        self.enricher = ThreatIntelEnricher(STIX_INDICATORS, THREAT_REPORTS)
        self.normalized: List[NormalizedEvent] = []
        self.incidents: List[SIEMIncident] = []
        self._incident_counter = 0

    def _new_id(self) -> str:
        self._incident_counter += 1
        return f"INC-{self._incident_counter:04d}"

    def ingest_raw_logs(self, raw_log_batch: List[Tuple[str, str]]):
        '''Stage 1: Normalize all raw log lines to CIM schema.'''
        for ts, line in raw_log_batch:
            self.normalizer_stats["total"] += 1
            ev = normalize(line, ts)
            if ev:
                self.normalized.append(ev)
                self.normalizer_stats["parsed"] += 1
            else:
                self.normalizer_stats["failed"] += 1

    def run_correlation(self):
        '''Stage 2: Run CEP correlation on normalized event stream.'''
        # Build SecurityEvent list for attack chain reconstructor
        base = time.time() - 3600
        sec_events = []
        for ev in self.normalized:
            parts = ev.timestamp.lstrip("T").split(":")
            h, m, s = int(parts[0]), int(parts[1]), int(parts[2])
            ts_offset = base + h * 3600 + m * 60 + s
            sec_events.append(SecurityEvent(
                ev.timestamp, ts_offset,
                ev.src_ip, ev.dst_ip, ev.dst_port,
                ev.action, ev.event_type, ev.source,
                f"{ev.event_type.upper()} {ev.action} {ev.src_ip}->{ev.dst_ip}:{ev.dst_port}",
                ev.username,
            ))

        # Also add the richer attack events from Demo 3
        sec_events.extend(all_security_events)
        self.reconstructor = AttackChainReconstructor(sec_events, window_seconds=3000)

        cep_alerts = []
        for ev in self.normalized:
            alert = self.cep_engine.ingest(ev)
            if alert:
                cep_alerts.append(alert)
        return cep_alerts

    def triage(self, cep_alerts: List[CorrelationAlert]) -> List[SIEMIncident]:
        '''Stages 3-5: Reconstruct chains, enrich with TI, generate LLM briefs.'''
        for alert in cep_alerts:
            # Stage 3: Find anchor event and reconstruct attack chain
            anchor = None
            for ts_f, ev in self.cep_engine.state[alert.entity]:
                if ev.event_type == "auth" and ev.outcome == "success":
                    anchor = ev
                    break
                if ev.event_type == "firewall" and ev.action == "allow":
                    anchor = ev

            chain_len = 1
            if anchor and self.reconstructor:
                sec_anchor = SecurityEvent(
                    anchor.timestamp,
                    self.cep_engine._ts_to_float(anchor.timestamp),
                    anchor.src_ip, anchor.dst_ip, anchor.dst_port,
                    anchor.action, anchor.event_type, anchor.source,
                    f"Anchor: {anchor.source} {anchor.action}",
                )
                chain = self.reconstructor.reconstruct(sec_anchor)
                chain_len = len(chain["timeline"])

            # Stage 4: Threat intelligence enrichment
            enrichment = self.enricher.enrich_alert(
                alert.entity, alert.description
            )
            ioc = enrichment.get("ioc_match")
            threat_actor = ioc["threat_actor"] if ioc else None
            ioc_conf = ioc["confidence"] if ioc else 0

            # Stage 5: LLM incident brief
            prompt = (
                f"SIEM incident summary:\n"
                f"Rule: {alert.rule_name} | Severity: {alert.severity}\n"
                f"Entity: {alert.entity} | Events: {alert.event_count}\n"
                f"Description: {alert.description}\n"
                f"Threat actor: {threat_actor or 'unknown'}\n"
                f"IoC confidence: {ioc_conf}%\n"
                f"Attack chain events: {chain_len}\n\n"
                f"Write 2-sentence analyst brief covering: "
                f"1) What is happening, 2) Immediate action required."
            )
            brief = llm_analyze(prompt, max_tokens=80)

            # Priority: 1=CRITICAL (IoC match + high score), 2=HIGH, 3=MEDIUM
            if ioc_conf >= 80 and alert.score >= 0.7:
                priority, severity = 1, "CRITICAL"
            elif ioc_conf >= 50 or alert.score >= 0.7:
                priority, severity = 2, "HIGH"
            else:
                priority, severity = 3, "MEDIUM"

            self.incidents.append(SIEMIncident(
                incident_id=self._new_id(),
                priority=priority,
                severity=severity,
                entity=alert.entity,
                rule_fired=alert.rule_name,
                event_count=len(alert.events),
                threat_actor=threat_actor,
                ioc_confidence=ioc_conf,
                attack_chain_length=chain_len,
                llm_brief=brief,
                mitre_tags=ioc["tags"] if ioc else [],
            ))

        # Sort by priority then score
        self.incidents.sort(key=lambda i: i.priority)
        return self.incidents

    def print_incident_queue(self):
        '''Print the analyst incident queue — prioritized and enriched.'''
        print("\n" + "=" * 70)
        print("ANALYST INCIDENT QUEUE — PRIORITIZED (Human approval required for all actions)")
        print("=" * 70)
        if not self.incidents:
            print("  No incidents generated.")
            return
        for inc in self.incidents:
            sev_label = {1:"[CRITICAL]",2:"[HIGH]   ",3:"[MEDIUM] "}.get(inc.priority,"[LOW]    ")
            actor = inc.threat_actor or "Unknown actor"
            print(f"\n{sev_label} {inc.incident_id} | {inc.entity} | {inc.rule_fired}")
            print(f"  Threat actor: {actor} | IoC confidence: {inc.ioc_confidence}%")
            print(f"  Chain events: {inc.attack_chain_length} | Tags: {inc.mitre_tags}")
            print(f"  Brief: {inc.llm_brief}")
            print(f"  -> ACTION: Analyst must approve all response steps")

# ── Run the full pipeline ──────────────────────────────────────────────────────
print("=" * 70)
print("FULL SIEM PIPELINE — END TO END")
print("=" * 70)

pipeline = SIEMPipeline()

# Ingest all raw logs (same batch from Demo 1 + additional)
extended_raw_logs = raw_logs + [
    ("T03:10:12", "%ASA-4-106023: Deny tcp src outside:185.220.1.100/43240 dst inside:10.0.0.5/445"),
    ("T03:10:14", "%ASA-4-106023: Deny tcp src outside:185.220.1.100/43242 dst inside:10.0.0.5/139"),
    ("T03:11:20", "EventID=4625 Account=admin IpAddress=185.220.1.100 Status=0xC000006A"),
]

print("\n[Stage 1] Normalizing raw logs...")
pipeline.ingest_raw_logs(extended_raw_logs)
stats = pipeline.normalizer_stats
print(f"  Total: {stats['total']} | Parsed: {stats['parsed']} | Failed: {stats['failed']}")

print("\n[Stage 2] Running CEP correlation engine...")
cep_alerts = pipeline.run_correlation()
print(f"  CEP alerts fired: {len(cep_alerts)}")
for a in cep_alerts:
    print(f"  -> {a.rule_name} ({a.severity}) entity={a.entity} score={a.score:.2f}")

print("\n[Stages 3-5] Attack chain reconstruction + TI enrichment + LLM triage...")
incidents = pipeline.triage(cep_alerts)

pipeline.print_incident_queue()

print("\n" + "=" * 70)
print(f"PIPELINE COMPLETE")
print(f"  Raw events ingested:   {stats['total']}")
print(f"  Normalized:            {stats['parsed']}")
print(f"  CEP alerts:            {len(cep_alerts)}")
print(f"  Incidents in queue:    {len(incidents)}")
print(f"  Coverage:              100% of events analyzed automatically")
print(f"  Analyst time per inc:  ~2 min (vs 90 min manual)")
print("All actions await human approval — AI enriches, analyst decides.")


## Summary: What You Built

A complete **SIEM integration pipeline** with 5 layers:

| Layer | Class/Function | Key Output |
|-------|---------------|-----------|
| **Normalizer** | `normalize()` + 4 parsers | `NormalizedEvent` (CIM schema) |
| **CEP Engine** | `CEPEngine` | `CorrelationAlert` when patterns fire |
| **Attack Chain** | `AttackChainReconstructor` | Full temporal timeline around anchor |
| **TI Enrichment** | `ThreatIntelEnricher` | IoC match + RAG threat report context |
| **LLM Triage** | `SIEMPipeline.triage()` | Prioritized `SIEMIncident` queue |

### Key Formulas

```
CEP state machine:  persistent rule + flowing events -> fires when pattern completes
Attack chain:       anchor event +/- time window + lateral pivot expansion
IoC enrichment:     exact hash lookup + RAG keyword search in threat reports
Priority score:     IoC confidence >= 80% + CEP score >= 0.7 -> CRITICAL
```

### Storage Architecture (Production)

```
Hot tier (recent 30 days):   Elasticsearch (LSM-Tree + inverted index) -> fast search
Cold tier (90-365 days):     Apache Parquet (column-oriented) -> 10-50x compression
CEP engine:                  Apache Kafka + Flink for real-time stream processing
Threat intel feed:           STIX/TAXII subscription -> vector index for RAG
```

### The Non-Negotiable Rule (Again)

> **Every AI recommendation requires explicit human approval.**
> The pipeline delivers context. The analyst makes the decision.
> A hallucinated block of a legitimate internal IP causes an outage.
> Human oversight is not optional — it is the architecture.

**Next: Chapter 75 — Network Anomaly Detection** — statistical baseline models
for traffic volume, protocol distribution shifts, and autoencoder-based
detection of multi-dimensional anomalies invisible to threshold rules.

In [ ]:
# ── Full Integration: Production SIEM Deployment Checklist ────────────────────
print("CHAPTER 72: PRODUCTION SIEM DEPLOYMENT CHECKLIST")
print("=" * 65)

checklist = [
    ("Data Collection",   "Cisco ASA / Juniper SRX syslog -> UDP 514 collector"),
    ("Data Collection",   "Windows Event Forwarding (WEF) -> WEC server"),
    ("Data Collection",   "NetFlow/IPFIX from core/distribution routers"),
    ("Data Collection",   "Linux auditd / syslog -> syslog-ng or rsyslog"),
    ("Normalization",     "CIM field mapping per vendor: src_ip, dst_ip, action, timestamp"),
    ("Normalization",     "LLM-assisted parser generation for new log sources"),
    ("Normalization",     "Schema validation: reject events missing required CIM fields"),
    ("CEP Rules",         "Brute force: 5+ auth failures in 5min from same IP"),
    ("CEP Rules",         "Port scan: 5+ firewall denies from same IP, >3 unique ports"),
    ("CEP Rules",         "Impossible travel: auth success >1000km apart in <1hr"),
    ("CEP Rules",         "New admin account: privilege escalation event outside change window"),
    ("TI Enrichment",     "STIX/TAXII feed subscription (MISP, ISAC, commercial feeds)"),
    ("TI Enrichment",     "Vector-indexed threat reports for RAG enrichment"),
    ("TI Enrichment",     "IoC cache with 24hr TTL (stale IoCs cause false positives)"),
    ("LLM Integration",   "Model: claude-haiku-4-5-20251011 (fast triage)"),
    ("LLM Integration",   "Always show evidence chain, not just verdict"),
    ("Guardrails",        "ALL response actions require analyst approval"),
    ("Guardrails",        "Immutable audit log: every AI recommendation + outcome"),
    ("Guardrails",        "False positive feedback loop -> rule and model improvement"),
]

current = ""
for category, item in checklist:
    if category != current:
        print(f"\n[{category}]")
        current = category
    print(f"  checkmark {item}")

print("\n" + "=" * 65)
print("ARCHITECTURE SUMMARY")
print("=" * 65)
print("Raw logs  ->  CIM Normalize  ->  CEP Correlate  ->  Chain Expand")
print("       ->  TI Enrich  ->  LLM Brief  ->  Analyst Queue")
print("")
print("Before: 50,000 raw events/day | 0.2% analyst coverage")
print("After:  50,000 events triaged | 100 prioritized incidents | 100% coverage")
print("")
print("Every action awaits human approval. AI enriches. Analyst decides.")
print("Chapter 72 complete.")
